# 01 - Scraping listings from 2nabsh

Collects residential sale listings from 2nabsh.com street by street across Tehran and turns them
into a table for modelling.

It is built to be re-run often, because Iranian property prices move quickly under high inflation
and a valuation dataset goes stale within weeks.

## Steps

1. Load a JSON file of street names and portal URLs.
2. Split the streets into 25 batches so the crawl can be run and resumed in chunks. Batches
   already written to disk are skipped.
3. For each street page, scroll until the loading element disappears, then collect listing URLs.
   Listings already in the portal archive are skipped.
4. For each listing, extract 53 fields.
5. Concatenate all batches, clean, deduplicate, one-hot encode.

## Fields

| Group | Fields |
|---|---|
| Price | price per m2 in million toman (target) |
| Physical | floor area, land area, total floors, floor, units per floor, bedrooms, building age |
| Categorical | unit type, deed type, orientation, facade, flooring, cabinets, neighbourhood |
| Listing | listing age in days, street, URL |
| Amenities | 33 binary flags |

Amenity flags: lift, parking, storage, fitted wardrobe, paint, terrace, video intercom, security
door, CCTV, electric gate, western WC, wallpaper, gas hob, panelling, extractor hood, master
bathroom, pool, sauna, jacuzzi, lobby, roof garden, gym, function room, evaporative cooler,
heater, package boiler, radiator, water heater, air conditioning, central heating, underfloor
heating, air handling unit, chiller.

## Cleaning rules

Fields arrive as free text and need normalising:

* `نوساز` (newly built) to building age 0
* `همکف` (ground floor) to floor 0
* `تک واحد` (single unit) to 1 unit per floor
* Relative dates to days: today and hours to 0, yesterday to 1, weeks x 7, months x 30
* Unit suffixes stripped from numeric strings
* Missing unit type defaults to `عادی` (standard)

In the encoding step, nulls are relabelled per column before one-hot encoding so that a missing
facade and a missing deed type do not end up in the same null category.

## Stack

Selenium (headless Chrome), BeautifulSoup, pandas, scikit-learn, openpyxl.


In [ ]:
# Colab setup: headless Chrome for Selenium. Skip if running locally with
# chromedriver already on PATH.
!apt-get update
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium
!pip install beautifulsoup4

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from selenium.webdriver.remote.errorhandler import TimeoutException
from selenium import webdriver
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import openpyxl
import requests ,random , time , json , os , sys

In [ ]:
# Load the JSON file containing the pre-collected list of street names and their URLs.


with open('street_links_raw.json') as f:
    street_links_raw = json.load(f)


In [ ]:
# Split the streets into 25 batches so the crawl can be run and resumed in chunks.

array = np.array_split(street_links_raw,25) # split by 25
splited_street = {}
for enum,k in enumerate(array):
    enum +=1
    ls = k.tolist()
    first , last = ls[0]['street_name'] , ls[-1]['street_name']
    splited_street[f'{first}_{last}({enum}).json'] = ls

In [ ]:
# Crawler. Handles three page types: region, single listing (case), and street.
# Street and region pages lazy-load, so we scroll until the 'loading' element disappears.
# Listings already moved to the archive are skipped.

def crawler(site,region = False , case = False,street = False):
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.83 Safari/537.36"
    sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
    options = webdriver.ChromeOptions()
    options.headless = True
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
        
    driver = webdriver.Chrome('chromedriver',options=options)
    driver.get(site)
    
    
    if region:
        try:
            condition = True
            while condition:
                source = driver.page_source
                soup = BeautifulSoup(source , 'html.parser')
                if soup.find('p',class_ = 'h3 mr-8'):
                    time.sleep(2)
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight-1000);")
                else:
                    condition = False

            time.sleep(2)
            driver.close()
            return soup
        except:
            print('An error occurred!,\nPlease try again')
        finally:
            driver.quit()
            
            
    elif case:
        try: 
            source = driver.page_source
            soup = BeautifulSoup(source , 'html.parser')
            time.sleep(1)
            driver.close()
            return soup
        except:
            print('An error occurred!,\nPlease try again')
        finally:
            time.sleep(1)
            driver.quit()
            
            
    elif street:
        try:
            condition = True
            while condition:
                source = driver.page_source
                soup = BeautifulSoup(source , 'html.parser')
                if soup.find('span',class_ = 'text-primary h4-normal mt-8'): #if archive exist
                    break
                if soup.find('p',class_ = 'h3 mr-8'):
                    time.sleep(2)
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight-1000);")
                else:
                    condition = False

            time.sleep(5)
            driver.close()
            return soup
        except:
            print('An error occurred!,\nPlease try again')
        finally:
            driver.quit()
            

    

In [ ]:
# Collect every listing URL for each street, skipping archived listings.

def getting_all_links(street_links):
    for street in street_links:
        if street['street_name'] == 'رودهن':
            continue
        s_soup = crawler(street['street_link'],street=True)
        link_div = s_soup.find_all('div',class_ = 'col-md-6 mt-md-16')
        links = []
        for div in link_div:
            # ignoring the archive cases
            if div.find('span',class_ = 'text-primary h4-normal mt-8'):
                continue
            links.append('https://www.2nabsh.com'+div.a['href'])

        street['cases_link'] = links
        street['cases_number'] = len(links)
        street_name = street['street_name']
        street_cases_number = street['cases_number']
        print(f'Street : { street_name } - Total : { street_cases_number } >>>  DONE' )
    return street_links


In [ ]:
# Collect listing URLs batch by batch and save each batch to its own JSON file.
# Skips batches that have already been written, so the crawl is resumable.

for i in splited_street:
    if not i in os.listdir(): # if the batch has not been scraped yet
        this_street = getting_all_links(splited_street[i])
        with open(i,'w') as f:
            json_object = json.dumps(this_street, indent=4)
            f.write(json_object)
        print(f'{i} DONE')
    else:
        continue

In [ ]:
cols_str='''لینک آگهی
شماره آگهی
نام آگهی در سایت
قدمت آگهی (واحد = روز)
نام خیابان
قیمت هر متر (واحد = میلیون تومان)
زیربنا
متراژ زمین
طبقات
طبقه
هر طبقه
نوع واحد
خواب
سند
سن بنا
موقعیت
نما
کف\u200cپوش
کابینت
تعداد امکانات
آسانسور
پارکینگ
انباری
کمد دیواری
نقاشی
تراس
آیفون تصویری
درب ضد سرقت
دوربین مداربسته
درب برقی
سرویس فرنگی
کاغذ دیواری
گاز رومیزی
پانل\u200cکوبی
هود
حمام مستر
استخر
سونا
جکوزی
لابی
روف گاردن
سالن ورزش
سالن اجتماعات
کولر آبی
بخاری
پکیج
رادیاتور
آبگرمکن
کولر گازی
شوفاژ
گرمایش از کف
هواساز
چیلر'''
cols_list = cols_str.split('\n')
cols_list
all_columns_title = {}
for count, c in enumerate(cols_list) :
    count += 1
    all_columns_title[c] = count

In [ ]:
main_informations_title_list = '''زیربنا
متراژ زمین
طبقات
طبقه
هر طبقه
نوع واحد
خواب
سند
سن بنا
موقعیت
نما
کف\u200cپوش
کابینت'''.split('\n')


In [ ]:
equipment_title_list = '''آسانسور
پارکینگ
انباری
کمد دیواری
نقاشی
تراس
آیفون تصویری
درب ضد سرقت
دوربین مداربسته
درب برقی
سرویس فرنگی
کاغذ دیواری
گاز رومیزی
پانل\u200cکوبی
هود
حمام مستر
استخر
سونا
جکوزی
لابی
روف گاردن
سالن ورزش
سالن اجتماعات
کولر آبی
بخاری
پکیج
رادیاتور
آبگرمکن
کولر گازی
شوفاژ
گرمایش از کف
هواساز
چیلر'''.split('\n')

In [ ]:
print('Batches collected so far:\n\n')
for i in os.listdir():
    if i[-5:] == '.json':
        if i == 'street_links_raw.json':
            continue
        else:
            print(i[:-5])

In [ ]:
# Set the target batch below.
# Example: target_street = 'Shariati_Abuzar(1)'


target_street = ''


In [ ]:
# Scrape every listing in the selected batch and write it to an Excel file. 
# Crawler. Handles three page types: region, single listing (case), and street.
# Street and region pages lazy-load, so we scroll until the 'loading' element disappears.


user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.83 Safari/537.36"
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
options = webdriver.ChromeOptions()
options.headless = True
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')


if f'{target_street}.xlsx' not in os.listdir(): #if excel file does not exist
    
    
    #creating the xlsx file and writing the columns title 
    wb = openpyxl.Workbook()
    sheet = wb.active
    for count,i in enumerate(all_columns_title):
        count += 1
        new_col = sheet.cell(row = 1, column = count)
        new_col.value = i
    wb.save(f'{target_street}.xlsx')
    
    
    
    # open the json file  
    with open(f'{target_street}.json') as f:
        street_links = json.load(f)

        
        
    driver = webdriver.Chrome('chromedriver',options=options)
    row_column = 1
    for street in street_links:
        
        this_street_total_cases_number = street['cases_number']
        this_street_name = street['street_name']
        
        
        if this_street_total_cases_number == 0: # if street is empty
            print(f'\n{this_street_name} is empty')
            continue
        for case_link in street['cases_link']:
            try:
                row_column += 1
                try:
                    driver.get(case_link)
                    time.sleep(2)
                    source = driver.page_source
                    soup = BeautifulSoup(source , 'html.parser')
                    
                    main_div = soup.find('div',class_ = 'row')
                    this_case ={}

                    this_case['لینک آگهی'] = case_link
                    this_case['شماره آگهی'] = row_column - 1
                    this_case['نام آگهی در سایت'] = main_div.address.text

                    when = main_div.find('span',class_ = 'sc-1bo5vg3-7 iRAoza d-block').text
                    if 'امروز' == when:
                        this_case['قدمت آگهی (واحد = روز)'] = 0
                    elif 'دیروز' == when:
                        this_case['قدمت آگهی (واحد = روز)'] = 1
                    elif 'ساعت' in  when:
                        this_case['قدمت آگهی (واحد = روز)'] = 0
                    elif 'روز' in when:
                        this_case['قدمت آگهی (واحد = روز)'] = int(when[:-8])
                    elif 'هفته' in when:
                        this_case['قدمت آگهی (واحد = روز)'] = int(when[:-9])*7
                    elif 'ماه' in when:
                        this_case['قدمت آگهی (واحد = روز)'] = int(when[:-8])*30

                    street_name_div = soup.find('div',class_ = 'mt-56')
#                     street_name = street_name_div.find_all('div',class_ = 'sc-1j3cqpe-0 TKoBK d-flex align-items-center mb-0 mb-md-8')[-1].text[17:]

                    this_case['نام خیابان'] = this_street_name
                    this_case['قیمت هر متر (واحد = میلیون تومان)'] = main_div.find('span',class_ = 'mr-16').text[5:-7].replace('٫','.')

                    main_informations = {}
                    for div in main_div.find_all('div',class_ = 'oows7l-0 hSDCey mt-8 mt-lg-16 col-md-4'):
                        main_informations[div.find('span').text[:-2]] = div.find('span',class_ = 'font-weight-bold').text

                    if main_informations.get('سن بنا'):
                        if main_informations['سن بنا'] == 'نوساز':
                            main_informations['سن بنا'] = 0
                        elif 'سال ساخت' in main_informations['سن بنا']:
                            age = main_informations['سن بنا'].replace('سال ساخت','')
                            main_informations['سن بنا'] = age

                    if main_informations.get('طبقه'):
                        if main_informations['طبقه'] == 'همکف':
                            main_informations['طبقه'] = 0

                    if main_informations.get('هر طبقه'):
                        if main_informations['هر طبقه'] == 'تک واحد':
                            main_informations['هر طبقه'] = 1
                        elif 'واحد' in  main_informations['هر طبقه']:
                            floor = main_informations['هر طبقه'].replace('واحد','')
                            main_informations['هر طبقه'] = floor

                    if not main_informations.get('نوع واحد'):
                        main_informations['نوع واحد'] = 'عادی'
                    this_case['اطلاعات اصلی'] = main_informations

                    equipment = []
                    for div in main_div.find_all('div', class_ = 'bl3k3x-2 eaWSsG d-flex align-items-center mt-16 mt-lg-24 col-md-4'):
                        equipment.append(div.text)
                    this_case['تجهیزات و امکانات'] = equipment

                    this_case['تعداد امکانات'] = len(equipment)

                    for c_column,info in enumerate(this_case):
                        c_column +=1



                        ## main informations
                        if c_column == 7 :
                            for j in this_case[info]:
                                for z in all_columns_title:
                                    if j == z :

                                        new_cell = sheet.cell(row = row_column, column = all_columns_title[z])
                                        new_cell.value = this_case[info][j]

                        elif c_column == 8 :

                            available_equipment = this_case[info]
                            unavailable_equipment = [e for e in equipment_title_list if e not in available_equipment ]

                            for i in available_equipment:
                                new_cell = sheet.cell(row = row_column, column = all_columns_title[i])
                                new_cell.value = 1
                            for j in unavailable_equipment:
                                new_cell = sheet.cell(row = row_column, column = all_columns_title[j])
                                new_cell.value = 0               

                        elif c_column == 9 :

                            new_cell = sheet.cell(row = row_column, column = 20)
                            new_cell.value = this_case[info]

                        else:
                            new_cell = sheet.cell(row = row_column, column = c_column)
                            new_cell.value = this_case[info]

                    wb.save(f'{target_street}.xlsx')
                    b = f'{this_street_name} __ case {row_column - 1}'
                    print (b, end="\r")
                except:
                    print('An error occurred!,\nPlease try again')
            except : 
                print('\nAn error occurred!,1 case missed')
                continue
                
else:
    print(f'{target_street}.xlsx')



In [ ]:
# Concatenate every scraped street file into one dataset, then clean:
# - strip unit suffixes from numeric columns ('metre', 'floor', 'bedroom')
# - drop rows with no listing URL or no floor area
# - extract the neighbourhood name out of the address string

available_street = []
for i in os.listdir():
    if i[-5:] == '.xlsx':
        if i in ['first-1.xlsx','first-2.xlsx','second-1.xlsx','second-2.xlsx','raw_sample.xlsx']:
            continue
        available_street.append(i)
        
dfs_list = []
for f in available_street:
    new_df = pd.read_excel(f)
    dfs_list.append(new_df)
    
raw_sample = pd.concat(dfs_list,axis = 0,ignore_index=True) # cocating all of streets
raw_sample.drop('شماره آگهی',axis=1,inplace=True)


raw_sample['خواب'].fillna('null',inplace = True)  # filling the rows that have no 'خواب' value with 'null'
raw_sample.dropna(subset=['لینک آگهی'],inplace = True) # deleting the rows without 'لینک آگهی'
raw_sample.dropna(subset=['زیربنا'],inplace = True) # deleting the rows without 'زیربنا'




raw_sample['زیربنا'] = raw_sample['زیربنا'].str.replace('متر','')
raw_sample = raw_sample.astype({"زیربنا": int}, errors='raise')

raw_sample['طبقات'] = raw_sample['طبقات'].str.replace('طبقه','')
raw_sample = raw_sample.astype({"طبقات": int}, errors='ignore')

raw_sample['خواب'] = raw_sample['خواب'].str.replace('خوابه','')

val = raw_sample['نام آگهی در سایت']
region_list = []
for i in raw_sample['نام آگهی در سایت'].values:
    this_region = i.split('،')
    region_list.append(this_region[2])

    


raw_sample.insert(4, 'نام محله',region_list)

raw_sample.to_excel('raw_sample.xlsx',index_label='شماره آگهی') # all excel files to raw_sample.xlsx


In [ ]:
df =  pd.read_excel('raw_sample.xlsx')

In [ ]:
## Drop duplicate listings (same listing URL).
df.drop_duplicates(subset = ['لینک آگهی'],inplace = True)


In [ ]:
# One-hot encode the 7 categorical fields (unit type, deed type, orientation,
# facade, flooring, cabinets, neighbourhood).
# Null values are relabelled per column so nulls from different columns don't collide
# into a single shared category.
df3 = df.copy()
df3.fillna('null',inplace=True)


categorical_title = ['نوع واحد','سند','موقعیت','نما','کف\u200cپوش','کابینت','نام محله']
ohe = OneHotEncoder()
feature_array = ohe.fit_transform(df3[categorical_title]).toarray()

feature_labels =  ohe.categories_
feature_label_list = []
for enum,i in enumerate(feature_labels):

    i = i.tolist()
    if 'null' in i:
        null_index = i.index('null')
        title = categorical_title[enum]
        i[null_index] = f'null{title}'

    feature_label_list.extend(i)

categorical =  pd.DataFrame(feature_array , columns = feature_label_list)
df3 = pd.concat([df3,categorical] , axis = 1)
df3.to_excel('second-1.xlsx',index=False)
